## 03 — Data Cleaning

> **Motivated and traceable transformation of the raw NBA game-level dataset.**  
> Every step is logged, reversible, and grounded in a decision documented in the Cleaning Plan.  
> **Input:** `data/1_raw/nba_stats_2000-01_2025-26.csv` (same hash as notebook 02).  
> **Output:** `data/2_processed/cleaned_nba_data_2000-01_2025-26.csv` — versioned clean dataset.

---

**Invariants**
- Raw data is never overwritten.
- Every transformation is logged with row/column counts before and after.
- No statistical imputation at this stage — that belongs to Feature Engineering.

### 1. Setup

In [ ]:
import hashlib
import json
import logging
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent. parent
sys.path.append(str(ROOT))

import polars as pl

from src.loader import load_config

config = load_config()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

NOTEBOOK_VERSION = "1.1.0"
log.info(f"Data cleaning notebook v{NOTEBOOK_VERSION} — initialized")

In [ ]:
start_year = config["settings"]["start_season"]
end_year   = config["settings"]["end_season"]

# Create labels for the start and end seasons.
start_season = f"{start_year}-{str(start_year + 1)[-2:]}"
end_season   = f"{end_year}-{str(end_year + 1)[-2:]}"

# Define the dataset version using the first and last season in the list.
DATASET_VERSION = f"{start_season}_{end_season}"

# Define the raw and processed data directories.
RAW_DIR       = ROOT / config["data"]["raw_data_dir"]
PROCESSED_DIR = ROOT / config["data"]["processed_data_dir"]

# Create the processed directory if it doesn't exist.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Define the path to the metadata file.
METADATA_FILE = RAW_DIR / f"metadata_{DATASET_VERSION}.json"

# Load the metadata from the file.
with open(METADATA_FILE, "r") as f:
    metadata = json.load(f)

RAW_FILE  = RAW_DIR / f"nba_stats_{start_season}_{end_season}.csv"
OUT_FILE  = PROCESSED_DIR / f"cleaned_nba_data_{start_season}_{end_season}.csv"

log.info(f"Raw input  : {RAW_FILE}")
log.info(f"Clean output: {OUT_FILE}")

#### 1.1 Load Raw Dataset — Hash Verification

We load the raw file with `infer_schema_length=0` (all columns as strings) to prevent Polars from silently coercing
mixed-type or dirty values before we have a chance to inspect and clean them.

The expected SHA-256 hash was recorded during `01_data_ingestion.ipynb` and is verified here to guarantee
we are operating on the exact same file.

In [ ]:
# Get the expected SHA-256 hash from the metadata
EXPECTED_SHA256 = metadata["sha256"]

with open(RAW_FILE, "rb") as f:
    actual_sha256 = hashlib.sha256(f.read()).hexdigest()

# Check if the actual hash matches the expected hash
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError(
        f"Hash mismatch!\n  Expected: {EXPECTED_SHA256}\n  Actual  : {actual_sha256}\n"
        "The raw file has changed since the assessment was run. Re-run notebook 02 first."
    )
log.info(f"Hash OK: {actual_sha256}")

# Load the raw dataset
df_raw = pl.read_csv(RAW_FILE, infer_schema_length=0)

INITIAL_ROWS = len(df_raw)
INITIAL_COLS = len(df_raw.columns)
log.info(f"Raw dataset loaded: {INITIAL_ROWS:,} rows × {INITIAL_COLS} columns")

### 2. Cleaning Plan

This section outlines the transformations applied to the raw dataset before feature engineering and modelling. The decisions below are based on the findings documented in `02_data_assessment.ipynb`.

#### 2.1 Summary of Cleaning Decisions

| Area                  | Decision                                                                            | Reason                                                                                                   |
| --------------------- | ----------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------- |
| Duplicate rows        | Remove one row per `(game_id, team_id)` pair after dropping lineup-specific columns | The BoxScore API stores separate rows for Starters and Bench statistics, creating structural duplicates. |
| Redundant columns     | Drop unused, duplicated, metadata, and derivable columns                            | Reduce schema complexity and retain only relevant information.                                           |
| Home/Away information | Create `home_away` from `MATCHUP`, then drop `MATCHUP`                              | Venue information is useful; the original string is not.                                                 |
| Percentage columns    | Drop `FG_PCT`, `FG3_PCT`, and `FT_PCT`                                              | Fully derivable from made and attempted shots.                                                           |
| Historical team codes | Keep as originally recorded                                                         | They correctly represent franchises at the time of the game.                                             |
| Extreme values        | Keep all records                                                                    | Assessment confirmed that extreme performances correspond to real NBA events.                            |
| Data types            | Cast columns to appropriate types                                                   | The raw dataset is loaded entirely as strings.                                                           |
| Invalid records       | Remove only physically impossible observations                                      | Data corruption should not be confused with legitimate outliers.                                         |

#### 2.2 Cleaning Workflow

The cleaning process follows these steps:

1. Extract `home_away` from `MATCHUP`.
2. Remove redundant, metadata, and lineup-specific columns.
3. Deduplicate team-game records.
4. Cast columns to their correct data types.
5. Rename columns using `snake_case`.
6. Reorder columns into a logical schema.
7. Validate numerical consistency:

   * `fgm <= fga`
   * `fg3m <= fg3a`
   * `ftm <= fta`
   * `pts > 0`
8. Standardize categorical values and trim whitespace.
9. Remove records missing mandatory identifiers or target variables.

#### 2.3 Expected Result

After cleaning:

* One row represents one team in one game.
* The schema contains only modelling-relevant variables.
* All columns have appropriate data types.
* No structural duplicates remain.
* Historical franchise information is preserved.
* Only valid basketball observations are retained.


### 3. Schema Enforcement

#### 3.1 Column Selection

**Target schema** — 19 original columns + 1 derived column (`home_away`):

| Final Name | Source Column | Category | Motivation |
|------------|--------------|----------|------------|
| `game_id` | `gameId` | Business key | Unique game identifier |
| `game_date` | `GAME_DATE` | Temporal key | Required for time-series splits and rolling windows |
| `season` | `SEASON` | Context | Season label for stratified splits |
| `team_id` | `teamId` | Entity key | Stable numeric team identifier |
| `team_abbreviation` | `teamTricode` | Entity label | Human-readable team code; historical codes retained |
| `home_away` | derived from `MATCHUP` | Context / feature | Venue advantage is a significant predictor |
| `wl` | `WL` | **Target** | Win/Loss — primary classification target |
| `pts` | `PTS` | **Target / feature** | Team points — primary regression target (point differential) |
| `plus_minus` | `PLUS_MINUS` | **Target** | Point differential — secondary regression target |
| `off_rating` | `offensiveRating` | Feature | Offensive efficiency per 100 possessions |
| `def_rating` | `defensiveRating` | Feature | Defensive efficiency per 100 possessions |
| `net_rating` | `netRating` | Feature | Net efficiency = off_rating − def_rating |
| `pace` | `pace` | Feature | Possessions per 48 minutes; controls game-tempo context |
| `ts_pct` | `trueShootingPercentage` | Feature | Shooting efficiency accounting for 3-pointers and free throws |
| `fgm` | `FGM` | Validation | Used for TS_PCT consistency check; team total from LGF |
| `fga` | `FGA` | Validation | Used for TS_PCT consistency check; team total from LGF |
| `fg3m` | `FG3M` | Validation | Used for TS_PCT consistency check; team total from LGF |
| `fg3a` | `FG3A` | Validation | Used for TS_PCT consistency check; team total from LGF |
| `ftm` | `FTM` | Validation | Used for TS_PCT consistency check; team total from LGF |
| `fta` | `FTA` | Validation | Used for TS_PCT consistency check; team total from LGF |

In [ ]:
TARGET_SCHEMA_SOURCES = [
    "gameId", "GAME_DATE", "SEASON", "teamId", "teamTricode",
    "home_away",
    "WL", "PTS", "PLUS_MINUS",
    "offensiveRating", "defensiveRating", "netRating", "pace", "trueShootingPercentage",
    "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
]

DROP_STRUCTURAL_A = [
    "teamCity", "teamName", "teamSlug",
    "teamCity_adv", "teamName_adv", "teamTricode_adv", "teamSlug_adv", "minutes_adv",
    "SEASON_ID", "TEAM_ABBREVIATION", "TEAM_NAME",
    "MIN",
    "FG_PCT", "FG3_PCT", "FT_PCT",
    "MATCHUP",
]

DROP_STRUCTURAL_B = [
    "fieldGoalsMade", "fieldGoalsAttempted", "fieldGoalsPercentage",
    "threePointersMade", "threePointersAttempted", "threePointersPercentage",
    "freeThrowsMade", "freeThrowsAttempted", "freeThrowsPercentage",
    "reboundsOffensive", "reboundsDefensive", "reboundsTotal",
    "assists", "steals", "blocks", "turnovers", "foulsPersonal",
    "points",
    "startersBench",
    "minutes",
]

DROP_MODEL_IRRELEVANT = [
    "estimatedOffensiveRating", "estimatedDefensiveRating", "estimatedNetRating",
    "assistPercentage", "assistToTurnover", "assistRatio",
    "offensiveReboundPercentage", "defensiveReboundPercentage", "reboundPercentage",
    "estimatedTeamTurnoverPercentage", "turnoverRatio",
    "effectiveFieldGoalPercentage",
    "usagePercentage", "estimatedUsagePercentage",
    "estimatedPace", "pacePer40",
    "possessions", "PIE",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF",
]

log.info(f"Structural drops (A — API metadata):         {len(DROP_STRUCTURAL_A)} columns")
log.info(f"Structural drops (B — starters/bench split): {len(DROP_STRUCTURAL_B)} columns")
log.info(f"Model-irrelevant drops:                      {len(DROP_MODEL_IRRELEVANT)} columns")
log.info(f"Total planned drops:                         {len(DROP_STRUCTURAL_A) + len(DROP_STRUCTURAL_B) + len(DROP_MODEL_IRRELEVANT)} columns")
log.info(f"Target schema size:                          {len(TARGET_SCHEMA_SOURCES)} columns (incl. derived home_away)")

In [ ]:
df = df_raw.with_columns(
    pl.when(pl.col("MATCHUP").str.contains(r"vs\."))
      .then(pl.lit("Home"))
      .when(pl.col("MATCHUP").str.contains("@"))
      .then(pl.lit("Away"))
      .otherwise(pl.lit(None))
      .alias("home_away")
)

home_away_counts = df["home_away"].value_counts(sort=True)
null_home_away = df["home_away"].null_count()

log.info("home_away derived from MATCHUP:")
for row in home_away_counts.iter_rows(named=True):
    log.info(f"  {row['home_away']}: {row['count']:,}")
if null_home_away:
    log.warning(f"  NULL (MATCHUP could not be parsed): {null_home_away}")

# Note: a small imbalance between Home/Away counts (pre-dedup) is expected.
# Neutral-site games (All-Star, NBA Cup, select international games) may have
# both teams encoded with '@' in the API, resulting in slightly more Away rows.
# This is a known NBA API behaviour, not a data error.

In [ ]:
missing_a = [c for c in DROP_STRUCTURAL_A if c not in df.columns]
if missing_a:
    log.warning(f"Structural-A columns not found (already absent): {missing_a}")

# Drop the columns that are in the DROP_STRUCTURAL_A list
to_drop_a = [c for c in DROP_STRUCTURAL_A if c in df.columns]
df = df.drop(to_drop_a)

log.info(
    f"[DROP — Structural A] {len(to_drop_a)} columns removed: {to_drop_a}\n"
    f"  Schema: {INITIAL_COLS + 1} → {len(df.columns)} columns (incl. derived home_away)"
)

In [ ]:
missing_b = [c for c in DROP_STRUCTURAL_B if c not in df.columns]
if missing_b:
    log.warning(f"Structural-B columns not found (already absent): {missing_b}")

# Drop the columns that are in the DROP_STRUCTURAL_B list
to_drop_b = [c for c in DROP_STRUCTURAL_B if c in df.columns]
df = df.drop(to_drop_b)

log.info(
    f"[DROP — Structural B — Starters/Bench splits] {len(to_drop_b)} columns removed: {to_drop_b}\n"
    f"  Schema: → {len(df.columns)} columns"
)

In [ ]:
missing_m = [c for c in DROP_MODEL_IRRELEVANT if c not in df.columns]
if missing_m:
    log.warning(f"Model-irrelevant columns not found (already absent): {missing_m}")

# Drop the columns that are in the DROP_MODEL_IRRELEVANT list
to_drop_m = [c for c in DROP_MODEL_IRRELEVANT if c in df.columns]
df = df.drop(to_drop_m)

log.info(
    f"[DROP — Model-irrelevant] {len(to_drop_m)} columns removed: {to_drop_m}\n"
    f"  Schema: → {len(df.columns)} columns"
)

In [ ]:
EXPECTED_PRE_RENAME = [
    "gameId", "teamId", "teamTricode",
    "offensiveRating", "defensiveRating", "netRating", "trueShootingPercentage", "pace",
    "SEASON", "GAME_DATE", "WL", "PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
    "PLUS_MINUS",
    "home_away",
]

# Verify that all expected columns are present
actual_cols = set(df.columns)
expected_cols = set(EXPECTED_PRE_RENAME)

extra   = actual_cols - expected_cols
missing = expected_cols - actual_cols

assert not extra,   f"Unexpected columns after drop: {extra}"
assert not missing, f"Missing expected columns after drop: {missing}"

log.info(f"Schema assertion passed: {len(df.columns)} columns present, all expected.")
log.info(f"Current schema: {df.columns}")

### 4. Deduplication

**Root cause of duplicates:** The NBA Stats API boxscore endpoint (`BoxScoreTraditionalV3`, `BoxScoreAdvancedV3`)
returns **two rows per team per game** — one for Starters, one for Bench. After dropping the split-specific
traditional stats (Section 3.1, Category B), all remaining columns contain **team-level values identical
across both rows** (advanced stats and LeagueGameFinder columns are not split by lineup group).

**Expected result:** 125,012 rows → 62,506 rows (exactly half, one per `(gameId, teamId)` pair).

**Strategy:** Drop exact duplicate rows (all 20 columns identical). No tiebreaker needed since rows
are fully identical after structural column removal.

In [ ]:
rows_before_dedup = len(df)

df = df.unique(maintain_order=True)

rows_after_exact_dedup = len(df)
exact_dupes_removed = rows_before_dedup - rows_after_exact_dedup

log.info(
    f"[DEDUP — Exact rows] {exact_dupes_removed:,} rows removed "
    f"({rows_before_dedup:,} → {rows_after_exact_dedup:,})"
)

In [ ]:
business_key = ["gameId", "teamId"]
n_unique_bk = df.select(business_key).unique().shape[0]
bk_dupes = len(df) - n_unique_bk

if bk_dupes > 0:
    log.warning(
        f"[DEDUP — Business key] {bk_dupes:,} rows still duplicate on {business_key}. "
        f"Investigate before proceeding."
    )
    sample = (
        df.filter(df.select(business_key).is_duplicated())
          .sort(business_key)
          .head(6)
    )
    display(sample)
else:
    log.info(
        f"[DEDUP — Business key] No remaining duplicates on {business_key}. "
        f"Each (gameId, teamId) pair is unique ({len(df):,} rows)."
    )

#### 4.1 Type Casting

All values were loaded as strings. We now cast each column to its correct physical type.

| Column | Target Type | Notes |
|--------|------------|-------|
| `gameId` | `String` | Structured ID — leading zeros must be preserved |
| `teamId` | `String` | Numeric-looking ID but used as categorical key |
| `teamTricode` | `String` | Categorical |
| `GAME_DATE` | `Date` | ISO format `YYYY-MM-DD` |
| `SEASON` | `String` | Categorical label (`2000-01`) |
| `home_away` | `String` | Binary categorical (`Home` / `Away`) |
| `WL` | `String` | Binary categorical (`W` / `L`) |
| `PTS` | `Int32` | Team total points |
| `PLUS_MINUS` | `Float32` | Point differential (can be fractional in API) |
| `offensiveRating` | `Float32` | Advanced team stat |
| `defensiveRating` | `Float32` | Advanced team stat |
| `netRating` | `Float32` | Advanced team stat |
| `pace` | `Float32` | Advanced team stat |
| `trueShootingPercentage` | `Float32` | Shooting efficiency [0, 1] |
| `FGM`, `FGA`, `FG3M`, `FG3A`, `FTM`, `FTA` | `Int32` | Counting stats — team totals from LGF |

In [ ]:
df = df.with_columns([
    pl.col("GAME_DATE").str.to_date("%Y-%m-%d"),
    pl.col("PTS").cast(pl.Int32),
    pl.col("PLUS_MINUS").cast(pl.Float32),
    pl.col("offensiveRating").cast(pl.Float32),
    pl.col("defensiveRating").cast(pl.Float32),
    pl.col("netRating").cast(pl.Float32),
    pl.col("pace").cast(pl.Float32),
    pl.col("trueShootingPercentage").cast(pl.Float32),
    pl.col("FGM").cast(pl.Int32),
    pl.col("FGA").cast(pl.Int32),
    pl.col("FG3M").cast(pl.Int32),
    pl.col("FG3A").cast(pl.Int32),
    pl.col("FTM").cast(pl.Int32),
    pl.col("FTA").cast(pl.Int32),
])

log.info("Type casting complete.")
log.info(f"Schema dtypes: {dict(zip(df.columns, [str(t) for t in df.dtypes]))}")

#### 4.2 Rename and Reorder

Rename all columns to `snake_case` and reorder into logical groups:
1. **Identifiers:** `game_id`, `game_date`, `season`
2. **Team:** `team_id`, `team_abbreviation`
3. **Context:** `home_away`, `wl`
4. **Targets:** `pts`, `plus_minus`
5. **Advanced ratings:** `off_rating`, `def_rating`, `net_rating`, `pace`, `ts_pct`
6. **Validation stats:** `fgm`, `fga`, `fg3m`, `fg3a`, `ftm`, `fta`

In [ ]:
RENAME_MAP = {
    "gameId":                 "game_id",
    "GAME_DATE":              "game_date",
    "SEASON":                 "season",
    "teamId":                 "team_id",
    "teamTricode":            "team_abbreviation",
    "WL":                     "wl",
    "PTS":                    "pts",
    "PLUS_MINUS":             "plus_minus",
    "offensiveRating":        "off_rating",
    "defensiveRating":        "def_rating",
    "netRating":              "net_rating",
    "trueShootingPercentage": "ts_pct",
    "pace":                   "pace",
    "FGM":                    "fgm",
    "FGA":                    "fga",
    "FG3M":                   "fg3m",
    "FG3A":                   "fg3a",
    "FTM":                    "ftm",
    "FTA":                    "fta",
}

df = df.rename(RENAME_MAP)

COLUMN_ORDER = [
    "game_id", "game_date", "season",
    "team_id", "team_abbreviation",
    "home_away", "wl",
    "pts", "plus_minus",
    "off_rating", "def_rating", "net_rating", "pace", "ts_pct",
    "fgm", "fga", "fg3m", "fg3a", "ftm", "fta",
]

df = df.select(COLUMN_ORDER)

log.info(f"Renamed {len(RENAME_MAP)} columns to snake_case.")
log.info(f"Column order: {df.columns}")
display(df.head(3))

### 5. Invalid Records Removal

Remove records that violate physical or logical constraints of basketball statistics.
These are **not** outliers — they indicate corrupt or malformed data.

| Rule | Column(s) | Condition | Rationale |
|------|----------|-----------|----------|
| R1 | `pts` | `pts <= 0` | A team must score at least 1 point to complete a valid game |
| R2 | `fgm`, `fga` | `fgm > fga` | Cannot make more field goals than attempted |
| R3 | `fg3m`, `fg3a` | `fg3m > fg3a` | Cannot make more 3-pointers than attempted |
| R4 | `ftm`, `fta` | `ftm > fta` | Cannot make more free throws than attempted |
| R5 | `fg3a`, `fga` | `fg3a > fga` | 3-point attempts cannot exceed total field goal attempts |
| R6 | `game_id`, `team_id` | `null` | Null business key — record is unidentifiable |
| R7 | `wl` | `null` | Null target variable — unusable for supervised learning |

In [ ]:
rows_before_invalid = len(df)
removal_log: dict[str, int] = {}

rules = {
    "R1_pts_nonpositive":    pl.col("pts") <= 0,
    "R2_fgm_gt_fga":         pl.col("fgm") > pl.col("fga"),
    "R3_fg3m_gt_fg3a":       pl.col("fg3m") > pl.col("fg3a"),
    "R4_ftm_gt_fta":         pl.col("ftm") > pl.col("fta"),
    "R5_fg3a_gt_fga":        pl.col("fg3a") > pl.col("fga"),
    "R6_null_business_key":  pl.col("game_id").is_null() | pl.col("team_id").is_null(),
    "R7_null_wl":            pl.col("wl").is_null(),
}

for rule_name, condition in rules.items():
    flagged = df.filter(condition)
    n = len(flagged)
    removal_log[rule_name] = n
    if n > 0:
        log.warning(f"[INVALID] {rule_name}: {n:,} records violate constraint — will be removed")
        display(flagged.head(3))
    else:
        log.info(f"[INVALID] {rule_name}: 0 violations — OK")

combined_invalid = (
    pl.col("pts") <= 0
    | (pl.col("fgm") > pl.col("fga"))
    | (pl.col("fg3m") > pl.col("fg3a"))
    | (pl.col("ftm") > pl.col("fta"))
    | (pl.col("fg3a") > pl.col("fga"))
    | pl.col("game_id").is_null()
    | pl.col("team_id").is_null()
    | pl.col("wl").is_null()
)

df = df.filter(~combined_invalid)

rows_after_invalid = len(df)
invalid_removed = rows_before_invalid - rows_after_invalid

log.info(
    f"[INVALID] Total removed: {invalid_removed:,} records "
    f"({rows_before_invalid:,} → {rows_after_invalid:,})"
)
log.info(f"Breakdown: {removal_log}")

### 6. Text Standardization

Defensive normalization of all string-valued columns:
- Strip leading/trailing whitespace.
- Assert domain of categorical columns (`wl`, `home_away`).
- Document historical team abbreviations — retained as-is.

#### Historical Team Abbreviation Reference

The dataset spans 2000-01 to 2025-26 and includes franchises that relocated or renamed.
These codes are **historically accurate** and are intentionally preserved.

| Legacy Code | Current Code | Franchise | Change |
|-------------|-------------|----------|--------|
| `CHH` | `CHA` | Charlotte Hornets | Renamed/relocated 2002 → New Orleans, 2004 expansion Charlotte Bobcats (CHA) |
| `NJN` | `BKN` | Brooklyn Nets | Relocated from New Jersey, rebranded 2012 |
| `SEA` | `OKC` | Oklahoma City Thunder | Relocated from Seattle 2008 |
| `VAN` | `MEM` | Memphis Grizzlies | Relocated from Vancouver 2001 |
| `NOH` | `NOP` | New Orleans Pelicans | New Orleans Hornets 2002–2013 |
| `NOK` | `NOP` | New Orleans Pelicans | Oklahoma City/New Orleans Hornets (Katrina season 2005-06) |
| `NOP` | `NOP` | New Orleans Pelicans | Renamed from Hornets 2013 (current) |

In [ ]:
STRING_COLS = [c for c, t in zip(df.columns, df.dtypes) if t == pl.String]

df = df.with_columns([
    pl.col(c).str.strip_chars()
    for c in STRING_COLS
])
log.info(f"Whitespace stripped from {len(STRING_COLS)} string columns: {STRING_COLS}")

VALID_WL = {"W", "L"}
invalid_wl = df.filter(~pl.col("wl").is_in(list(VALID_WL)))
assert len(invalid_wl) == 0, f"Unexpected WL values: {df['wl'].unique().to_list()}"
log.info(f"wl domain OK: {sorted(df['wl'].unique().to_list())}")

VALID_HOME_AWAY = {"Home", "Away"}
invalid_ha = df.filter(~pl.col("home_away").is_in(list(VALID_HOME_AWAY)))
assert len(invalid_ha) == 0, f"Unexpected home_away values: {df['home_away'].unique().to_list()}"
log.info(f"home_away domain OK: {sorted(df['home_away'].unique().to_list())}")

HISTORICAL_TRICODES = {"CHH", "NJN", "SEA", "VAN", "NOH", "NOK"}
ACTIVE_TRICODES = set(df["team_abbreviation"].unique().to_list()) - HISTORICAL_TRICODES
historical_present = set(df["team_abbreviation"].unique().to_list()) & HISTORICAL_TRICODES

log.info(f"Total unique team abbreviations: {df['team_abbreviation'].unique().len()}")
log.info(f"Historical tricodes present (retained intentionally): {sorted(historical_present)}")
log.info(f"Active tricodes: {sorted(ACTIVE_TRICODES)}")

### 7. Missing Value Handling

**Policy:**
1. Drop any column that is entirely null or below a utility threshold (> 30% null) — none expected at this stage.
2. Drop any record where a mandatory field is null. Mandatory fields:
   - Business key: `game_id`, `team_id`
   - Target variable: `wl`
   - Context: `game_date`, `season`, `home_away`
3. Records with null advanced stats (off_rating, def_rating, net_rating, pace, ts_pct) are logged but not removed — imputation belongs to Feature Engineering.

**No statistical imputation is performed here.** Any imputation strategy (e.g., rolling-window mean fill) requires knowledge of the model context and must be decided in notebook `04_feature_engineering`.

In [ ]:
rows_before_missing = len(df)

null_counts = {
    col: df[col].null_count()
    for col in df.columns
}
null_summary = {k: v for k, v in null_counts.items() if v > 0}

if null_summary:
    log.warning(f"Columns with nulls: {null_summary}")
else:
    log.info("No null values detected in any column.")

HIGH_NULL_THRESHOLD = 0.30
for col, n_null in null_counts.items():
    pct = n_null / len(df)
    if pct > HIGH_NULL_THRESHOLD:
        log.warning(f"  Column '{col}' is {pct:.1%} null — consider dropping in Feature Engineering.")

MANDATORY_FIELDS = ["game_id", "team_id", "wl", "game_date", "season", "home_away"]
mandatory_null_mask = pl.lit(False)
for field in MANDATORY_FIELDS:
    mandatory_null_mask = mandatory_null_mask | pl.col(field).is_null()

mandatory_violations = df.filter(mandatory_null_mask)
if len(mandatory_violations) > 0:
    log.warning(f"[MISSING] {len(mandatory_violations):,} records with null mandatory fields — removing.")
    df = df.filter(~mandatory_null_mask)
else:
    log.info("[MISSING] No records with null mandatory fields.")

ADVANCED_STATS = ["off_rating", "def_rating", "net_rating", "pace", "ts_pct"]
for stat in ADVANCED_STATS:
    n = df[stat].null_count()
    if n > 0:
        log.warning(
            f"[MISSING] '{stat}' has {n:,} nulls ({n/len(df):.3%}) — "
            "retained for Feature Engineering imputation decision."
        )

rows_after_missing = len(df)
log.info(
    f"[MISSING] Rows after missing-value handling: {rows_after_missing:,} "
    f"(removed {rows_before_missing - rows_after_missing:,})"
)

### 8. Cleaning Report

Full audit trail of all transformations applied in this notebook.

In [ ]:
FINAL_ROWS = len(df)
FINAL_COLS = len(df.columns)

total_removed = INITIAL_ROWS - FINAL_ROWS
pct_lost = total_removed / INITIAL_ROWS * 100

total_cols_dropped = (
    len(DROP_STRUCTURAL_A)
    + len(DROP_STRUCTURAL_B)
    + len(DROP_MODEL_IRRELEVANT)
)

print("=" * 60)
print("           CLEANING REPORT — NBA Dataset")
print("=" * 60)
print(f"  Raw input              : {INITIAL_ROWS:>10,} rows × {INITIAL_COLS} columns")
print(f"  Clean output           : {FINAL_ROWS:>10,} rows × {FINAL_COLS} columns")
print()
print("  ROW REMOVALS")
print(f"    Exact duplicates     : {exact_dupes_removed:>10,}  (Starters/Bench API split)")
print(f"    Invalid records      : {invalid_removed:>10,}  (physical constraint violations)")
print(f"    Missing mandatory    : {rows_before_missing - rows_after_missing:>10,}  (null business key or target)")
print(f"    Total rows removed   : {total_removed:>10,}  ({pct_lost:.2f}% of raw)")
print()
print("  COLUMN DROPS")
print(f"    Structural A (API metadata)          : {len(DROP_STRUCTURAL_A):>3}")
print(f"    Structural B (starters/bench splits) : {len(DROP_STRUCTURAL_B):>3}")
print(f"    Model-irrelevant                     : {len(DROP_MODEL_IRRELEVANT):>3}")
print(f"    Derived column added (home_away)     :  +1")
print(f"    Total net column change              : {FINAL_COLS - INITIAL_COLS:>+3} (from {INITIAL_COLS} to {FINAL_COLS})")
print()
print("=" * 60)

log.info(f"Cleaning complete. Final dataset: {FINAL_ROWS:,} rows × {FINAL_COLS} columns.")

In [ ]:
print("\n  FINAL NULL AUDIT")
final_nulls = {col: df[col].null_count() for col in df.columns}
null_cols = {k: v for k, v in final_nulls.items() if v > 0}
if null_cols:
    for col, n in null_cols.items():
        print(f"    {col}: {n:,} nulls ({n/FINAL_ROWS:.3%})")
else:
    print("    No null values in any column.")

print("\n  FINAL SCHEMA")
for col, dtype in zip(df.columns, df.dtypes):
    print(f"    {col:<20} {str(dtype)}")

### 9. Save Clean Dataset

In [ ]:
df.write_csv(OUT_FILE)

with open(OUT_FILE, "rb") as f:
    output_sha256 = hashlib.sha256(f.read()).hexdigest()

log.info(f"Clean dataset written to: {OUT_FILE}")
log.info(f"Output SHA-256: {output_sha256}")
log.info(f"Rows: {FINAL_ROWS:,}  |  Columns: {FINAL_COLS}")
log.info(f"Seasons covered: {df['season'].min()} – {df['season'].max()}")
log.info(f"Date range: {df['game_date'].min()} – {df['game_date'].max()}")

print("\n  OUTPUT DATASET SUMMARY")
print(f"    Path       : {OUT_FILE}")
print(f"    SHA-256    : {output_sha256}")
print(f"    Shape      : {FINAL_ROWS:,} rows × {FINAL_COLS} columns")
print(f"    Seasons    : {df['season'].min()} – {df['season'].max()}")
print(f"    Dates      : {df['game_date'].min()} – {df['game_date'].max()}")
print()
print("  COLUMN LIST")
for i, col in enumerate(df.columns, 1):
    dtype = df.schema[col]
    print(f"    {i:02d}. {col:<20} {str(dtype)}")

In [ ]:
display(df.head(5))